# ROMAN + MiniRocket on multivariate time series

This example uses EthanolConcentration from the UEA archive: three channels and 1,751 time steps. It compares MiniRocket on the original input with ROMAN S=4 followed by the same backbone. The paper reports mean accuracy of about 30.4% and 37.3%, respectively, over five seeds.

Here we run seed 1 as a compact worked example. Training includes the full preprocessing and model fit; inference is measured separately on the complete test batch after warm-up. Both methods use 10,000 kernels and one compute thread.

In [1]:
from pathlib import Path
import sys

# Support running the notebook from the repository root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "roman" / "__init__.py").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "notebooks"))

from IPython.display import display
from example_utils import (load_dataset, make_model, benchmark_single,
                           benchmark_ensembles, display_single, environment)


In [2]:
DATASET = "EthanolConcentration"
DATA_ROOT = None
X_train, y_train, X_test, y_test = load_dataset(DATASET, DATA_ROOT)
print("Training:", X_train.shape, "  Test:", X_test.shape)

from roman import RomanOperator
operator = RomanOperator(S=4, alpha=0.5)
Z_train = operator.fit_transform(X_train)
Z_test = operator.transform(X_test)
print("ROMAN training:", Z_train.shape, "  Test:", Z_test.shape)
print("Scale lengths:", operator.lengths_)
print("Windows per scale:", operator.windows_)

Training: (261, 3, 1751)   Test: (263, 3, 1751)
ROMAN training: (261, 78, 219)   Test: (263, 78, 219)
Scale lengths: [1751, 876, 438, 219]
Windows per scale: [15, 7, 3, 1]


In [3]:
details, summary = benchmark_single(
    X_train, y_train, X_test, y_test,
    seeds=(1,), scales=(1, 4), repeats=5,
)
display(display_single(summary))

S=1, seed=1: accuracy=31.94%


S=4, seed=1: accuracy=38.02%


,Model,Test accuracy,Training time,Inference time
0,MiniRocket (S=1),31.94%,2.334 s,2.191 s
1,ROMAN S=4 + MiniRocket,38.02%,1.052 s,0.966 s


The raw S=1 baseline and normalized ROMAN S=4 inputs follow the paper's MiniRocket example protocol. The result illustrates how the same operator handles multichannel inputs: original channels, scales and coarse temporal windows become explicit pseudochannels. Use validation data to choose S for a new dataset.

In [4]:
environment()

{'python': '3.12.14',
 'platform': 'macOS-27.2-arm64-arm-64bit',
 'machine': 'arm64',
 'threads': 1,
 'packages': {'numpy': '2.3.5',
  'pandas': '2.3.3',
  'scipy': '1.17.1',
  'scikit-learn': '1.7.2',
  'sktime': '1.1.0',
  'numba': '0.63.1',
  'threadpoolctl': '3.6.0'}}